# Поиск аномалий

In [10]:
import pandas as pd
import numpy as np
import re

In [11]:
pd.set_option("display.max_columns", None)

In [12]:
df = pd.read_csv("data/hakaton.csv", sep=";")
df

,last_name,first_name,middle_name,bdate,gender,id_doc,guard_last_name,guard_first_name,guard_middle_name,guard_bdate,guard_gender,guard_id_doc,our_number,ogrn_naprav,name_naprav,ogrn_area,name_area,variant,class,test_date,result
0,МУСЮС,АЛЕКСАНДР,NaN,2013-07-16,М,240045105,АГАФОНОВА,КЛАВДИЯ,NaN,1987-07-06,Ж,106077198701603,26-11-60078-000038-4,1022402487304,муниципальное автономное общеобразовательное у...,1142452001130,краевое бюджетное общеобразовательное учрежден...,20611,6,2026-02-25,Недостаточный
1,СОБОЛЕВ,ЮРИЙ,ГЕННАДЬЕВИЧ,2018-01-26,М,-1236233,КАБЛИН,ВАЛЕНТИН,ГЕННАДИЕВИЧ,1991-03-15,М,1571302,25-11-70122-000036-6,1025007390077,МУНИЦИПАЛЬНОЕ БЮДЖЕТНОЕ ОБЩЕОБРАЗОВАТЕЛЬНОЕ УЧ...,1035004258970,МУНИЦИПАЛЬНОЕ БЮДЖЕТНОЕ ОБЩЕОБРАЗОВАТЕЛЬНОЕ УЧ...,120126,1,2025-12-16,Недостаточный
2,РЯБОВ,ГЕННАДИЙ,СЕРГЕЕВИЧ,2014-08-21,М,5974549,ЗАКИРОВ,БОРИС,СЕРГЕЕВИЧ,1990-03-07,М,5480835,26-11-70285-000004-4,1037816029393,ГОСУДАРСТВЕННОЕ БЮДЖЕТНОЕ ОБЩЕОБРАЗОВАТЕЛЬНОЕ ...,1037816029547,Государственное бюджетное общеобразовательное ...,10410,4,2026-01-20,Достаточный
3,БОРИСОВ,АНАТОЛИЙ,НИКОЛЛАЕВИЧ,2018-07-22,М,3702355,ДЖАЛОЛОВ,ЕВГЕНИЙ,АНДРАНИКОВИЧ,1992-08-23,М,402238324,26-11-70046-000006-7,1021603470360,Муниципальное бюджетное общеобразовательное уч...,1241600037315,Муниципальное бюджетное общеобразовательное уч...,20133,1,2026-02-04,Недостаточный
4,КАРПЕВИЧ,ВИКТОР,ЮРЬЕВИЧ,2019-01-18,М,0000000,ШАПРАН,ЕЛЕНА,ПАВЛОВНА,1995-01-17,Ж,0352905,25-11-60108-000020-9,1023402988234,муниципальное общеобразовательное учреждение С...,1023402982173,"муниципальное общеобразовательное учреждение ""...",60105,1,2025-06-10,достаточный
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25623,ЛАЗАРЕВ,ДЕНИС,ВЛАДИМИРОВИЧ,2015-06-28,М,2118425,ЛАЗАРЕВ,ВАЛЕРИЙ,ПЕТРОВИЧ,1991-10-06,М,402450094,25-11-70289-000015-7,1022701291250,муниципальное автономное общеобразовательное у...,1022701289170,муниципальное бюджетное общеобразовательное уч...,60304,3,2025-07-14,Недостаточный
25624,ВИДАКОВ,МИХАИЛ,СЕРГЕЕВИЧ,2018-03-06,М,010080310,КРАСНОКУТСКИЙ,ИЛЬЯ,ВЛАДИМИРОВИЧ,1991-11-07,М,4127610,25-11-70289-000009-8,1022701286508,муниципальное автономное общеобразовательное у...,1022701289170,муниципальное бюджетное общеобразовательное уч...,80110,1,2025-08-27,Недостаточный
25625,ТОЛОКОНУШКОВ,ЕВГЕНИЙ,ФЕДОРОВИЧ,2018-04-14,М,-N0849879,ВАСИЛЕНКО,БОРИС,СЕРГЕЕВИЧ,1978-10-02,М,2920197,25-11-70289-000014-8,1022701283208,муниципальное автономное общеобразовательное у...,1022701289170,муниципальное бюджетное общеобразовательное уч...,80109,1,2025-08-27,Достаточный
25626,ТИХОНОВ,ВЛАДИМИР,АЛЕКСАНДРОВИЧ,2017-09-11,М,21109201751729,МИРЗОЯН,ВЛАДИМИР,АЛЕКСАНДРОВИЧ,1987-05-17,М,3196439,25-11-70289-000019-9,1022701291250,муниципальное автономное общеобразовательное у...,1022701289170,муниципальное бюджетное общеобразовательное уч...,90114,1,2025-09-25,Недостаточный


## Удаление дубликатов

In [13]:
df["result"] = df["result"].str.strip().str.upper()

cols_content = [c for c in df.columns if c not in ["our_number", "name_naprav", "name_area"]]
before = len(df)
df = df.drop_duplicates(subset=cols_content, keep="first")
print(f"  1a. Полные дубликаты удалены: {before - len(df)}")

  1a. Полные дубликаты удалены: 183


In [14]:
# 1c. Дубликаты по ключу (id_doc + test_date + variant + class), исключая id_doc=NaN/"-"
df["id_doc"] = df["id_doc"].replace("-", np.nan)
mask_valid = df["id_doc"].notna()

df_valid = df[mask_valid].copy()
df_invalid = df[~mask_valid].copy() 

before = len(df_valid)
df_valid = df_valid.sort_values(["id_doc", "test_date", "variant", "class", "result"])
df_valid = df_valid.drop_duplicates(subset=["id_doc", "test_date", "variant", "class"], keep="last")
print(f"  1b. Дубликаты по (id_doc, date, variant, class): {before - len(df_valid)}")

df = pd.concat([df_valid, df_invalid], ignore_index=True)
print(f"  Итого после дедупликации: {len(df)} записей\n")

  1b. Дубликаты по (id_doc, date, variant, class): 80
  Итого после дедупликации: 25365 записей



## НОРМАЛИЗАЦИЯ АТРИБУТОВ

In [15]:
for col in ["bdate", "test_date", "guard_bdate"]:
    df[col] = pd.to_datetime(df[col])
print("  2a. Даты приведены к datetime")

  2a. Даты приведены к datetime


In [16]:
# 2b. variant — извлекаем числовой код
def normalize_variant(v):
    if pd.isna(v):
        return v
    v = str(v).strip()

    # Уже числовой
    try:
        int(v)
        return v
    except ValueError:
        pass

    # "УЧ XXXXXX" или "УЧ XXXXXX,ПЧ XXXXXX"
    m = re.match(r"^УЧ\s+(\d+)", v)
    if m:
        return m.group(1)

    # "727.XXXXXX", "747.XXXXXX", разделители . _ /
    m = re.match(r"^7[24]7[._/](\d{5,7})$", v)
    if m:
        return m.group(1)

    # "727.110.508" ---> "110508"
    m = re.match(r"^727\.(\d+)\.(\d+)$", v)
    if m:
        return m.group(1) + m.group(2)

    # "№ XXXXXX"
    m = re.match(r"^№\s*(\d+)", v)
    if m:
        return m.group(1)

    # "0 XXXXX" (пробел)
    m = re.match(r"^0\s+(\d+)$", v)
    if m:
        return "0" + m.group(1)

    # "`XXXXXX" (бэктик-опечатка)
    m = re.match(r"^`(\d+)$", v)
    if m:
        return m.group(1)

    # "120.126" ---> "120126" (но не даты XX.XX.XXXX)
    m = re.match(r"^(\d{3,})\.(\d+)$", v)
    if m:
        return m.group(1) + m.group(2)

    return np.nan

In [17]:
df["variant_original"] = df["variant"]
df["variant"] = df["variant"].apply(normalize_variant)
recovered = (df["variant"].notna() & (df["variant"] != df["variant_original"])).sum()
lost = df["variant"].isna().sum()
print(f"  2b. variant: нормализовано {recovered}, не удалось распарсить {lost}")

  2b. variant: нормализовано 798, не удалось распарсить 25


In [18]:
# 2c. class
df.loc[df["class"] == "2-5", "class"] = np.nan
print(f"  2c. class='2-5' ---> NaN")

  2c. class='2-5' ---> NaN


In [19]:
# 2d. Убираем служебные столбцы (оставляем в отдельном df если нужно)
df_full = df.copy()  # полная копия со всеми столбцами
df = df.drop(columns=["our_number", "name_naprav", "name_area", "variant_original"], errors="ignore")
print(f"  2d. Убраны служебные столбцы\n")

  2d. Убраны служебные столбцы



## ВЫЯВЛЕНИЕ АНОМАЛИЙ

In [20]:
df["child_age"] = (df["test_date"] - df["bdate"]).dt.days / 365.25
df["parent_age_at_birth"] = (df["bdate"] - df["guard_bdate"]).dt.days / 365.25
class_num = pd.to_numeric(df["class"], errors="coerce")

# --- Флаги ---
flags = {}

# Пропуски
flags["id_doc_missing"] = df["id_doc"].isna()
flags["variant_invalid"] = df["variant"].isna()
flags["class_invalid"] = df["class"].isna() | (class_num.isna() & df["class"].notna())
flags["guard_id_missing"] = df["guard_id_doc"].isna()

# Даты
flags["test_before_birth"] = df["test_date"] < df["bdate"]
flags["bdate_equals_guard"] = df["bdate"] == df["guard_bdate"]
flags["child_too_young"] = df["child_age"] < 5
flags["child_too_old"] = df["child_age"] > 20
flags["guard_same_age"] = df["parent_age_at_birth"] <= 0
flags["guard_too_young_parent"] = (df["parent_age_at_birth"] > 0) & (df["parent_age_at_birth"] < 18)

# ID ребёнка = ID родителя
flags["id_equals_guard_id"] = (df["id_doc"] == df["guard_id_doc"]) & df["id_doc"].notna()

# Несоответствие возраст-класс (>3 года)
expected_age = class_num + 6
flags["age_class_mismatch"] = ((df["child_age"] - expected_age).abs() > 3) & expected_age.notna()

# Один ребёнок — несколько классов
valid_ids = df[df["id_doc"].notna()]
child_classes = valid_ids.groupby("id_doc")["class"].nunique()
multi_class_ids = child_classes[child_classes > 1].index
flags["multi_class"] = df["id_doc"].isin(multi_class_ids)

# Нарушения частоты (<90 дней)
df_s = df[df["id_doc"].notna()].sort_values(["id_doc", "test_date"]).copy()
df_s["prev_test"] = df_s.groupby("id_doc")["test_date"].shift(1)
df_s["days_gap"] = (df_s["test_date"] - df_s["prev_test"]).dt.days
freq_idx = df_s[df_s["days_gap"].notna() & (df_s["days_gap"] < 90)].index
same_day_idx = df_s[df_s["days_gap"] == 0].index
flags["freq_violation"] = df.index.isin(freq_idx)
flags["same_day_test"] = df.index.isin(same_day_idx)

# ОГРН некорректной длины
flags["ogrn_naprav_bad"] = df["ogrn_naprav"].astype(str).str.len() != 13
flags["ogrn_area_bad"] = df["ogrn_area"].astype(str).str.len() != 13

# --- Записываем флаги ---
for name, mask in flags.items():
    df[name] = mask

In [21]:
critical = [f for f in flags if f != "guard_id_missing"]
df["anomaly_count"] = sum(flags[f] for f in critical)
df["has_anomaly"] = df["anomaly_count"] > 0

# --- Вывод ---
print(f"\nВсего записей: {len(df)}")
print(f"С аномалиями: {df["has_anomaly"].sum()} ({df["has_anomaly"].mean()*100:.1f}%)")
print(f"Чистых: {(~df["has_anomaly"]).sum()} ({(~df["has_anomaly"]).mean()*100:.1f}%)")

print(f"\nТипы аномалий:")
for name in sorted(flags, key=lambda f: -flags[f].sum()):
    cnt = flags[name].sum()
    if cnt > 0:
        print(f"  {name}: {cnt}")

# --- Нарушения частоты ---
print(f"\nНарушения частоты (<90 дней):")
gaps = df_s["days_gap"].dropna()
freq = gaps[gaps < 90]
print(f"  Всего: {len(freq)} записей, {df_s[df_s["days_gap"].notna() & (df_s["days_gap"] < 90)]["id_doc"].nunique()} детей")
print(f"  В один день: {(freq == 0).sum()}")
print(f"  1-30 дней: {((freq >= 1) & (freq <= 30)).sum()}")
print(f"  31-60 дней: {((freq >= 31) & (freq <= 60)).sum()}")
print(f"  61-89 дней: {((freq >= 61) & (freq <= 89)).sum()}")


Всего записей: 25365
С аномалиями: 2281 (9.0%)
Чистых: 23084 (91.0%)

Типы аномалий:
  multi_class: 1042
  freq_violation: 832
  age_class_mismatch: 394
  id_equals_guard_id: 320
  guard_too_young_parent: 197
  same_day_test: 158
  guard_id_missing: 76
  id_doc_missing: 46
  variant_invalid: 25
  child_too_young: 6
  child_too_old: 6
  bdate_equals_guard: 3
  guard_same_age: 3
  class_invalid: 1
  test_before_birth: 1
  ogrn_naprav_bad: 1

Нарушения частоты (<90 дней):
  Всего: 832 записей, 703 детей
  В один день: 158
  1-30 дней: 195
  31-60 дней: 215
  61-89 дней: 264


In [22]:
df.to_csv("data/cleaned_with_flags.csv", index=False)
df_clean = df[~df["has_anomaly"]].drop(columns=list(flags.keys()) + ["anomaly_count", "has_anomaly", "child_age", "parent_age_at_birth"])
df_clean.to_csv("data/cleaned_final.csv", index=False)
print(f"\n[СОХРАНЕНО]")
print(f"  cleaned_with_flags.csv — все записи с флагами аномалий ({len(df)})")
print(f"  cleaned_final.csv — только чистые записи ({len(df_clean)})")


[СОХРАНЕНО]
  cleaned_with_flags.csv — все записи с флагами аномалий (25365)
  cleaned_final.csv — только чистые записи (23084)
